# UAVIDS-2025 — benchmark Docker local

Este notebook analisa a rodada final `docker_local_v2`. A pergunta é restrita: qual é o custo de servir os artefatos congelados de XGBoost e Random Forest por HTTP local, em containers separados com 0,5 CPU e 512 MiB?

CNN, GNN, stacking, Kubernetes, energia e rede entre máquinas estão fora do escopo. O protocolo foi congelado antes da v2 em [`docker_local_v2.md`](../../protocol/docker_local_v2.md).

## Por que existe uma versão 2?

O piloto v1 apresentou cerca de 45 ms constantes fora de `predict_proba`, compatíveis com Nagle/delayed ACK em mensagens pequenas. A v1 foi descartada, `TCP_NODELAY` foi registrado como correção e os dois modelos foram repetidos integralmente. Esta decisão está em [`deviations.md`](../../protocol/deviations.md); nenhum número da v1 deve entrar no artigo.

In [1]:
from pathlib import Path
import json
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "research":
    project_root = project_root.parents[1]
elif project_root.name == "notebooks":
    project_root = project_root.parent
benchmark_dir = project_root / "benchmarks" / "docker_local_v2"
report_dir = project_root / "reports" / "docker_local_v2"
manifest = json.loads((benchmark_dir / "manifest.json").read_text("utf-8"))
print({
    "status": manifest["status"],
    "image_id": manifest["image_id"],
    "docker": manifest["docker_version"]["Server"]["Platform"]["Name"],
    "kernel": manifest["docker_version"]["Server"]["KernelVersion"],
    "models": manifest["models"],
})

{'status': 'complete', 'image_id': 'sha256:a98c32d1bb0d09c3b4441d2b95fff83e93c130978d5f5cba1461cf32aff9a5c3', 'docker': 'Docker Desktop 4.66.1 (222799)', 'kernel': '6.6.114.1-microsoft-standard-WSL2', 'models': ['xgboost', 'random_forest']}


## Latência HTTP individual

O cronômetro do cliente começa com o corpo JSON já serializado e termina após a leitura integral da resposta. Portanto, inclui transporte local, parsing no servidor, fila, inferência e serialização da resposta. O tempo interno de `predict_proba` também foi registrado separadamente.

In [2]:
latency = pd.read_csv(report_dir / "individual_http_latency.csv")
print(latency.round(3).to_string(index=False))

        model  model_mib  count   mean_us    std_us  p50_us    p95_us     p99_us   max_us  server_predict_p50_us  server_predict_p99_us
      xgboost      2.756   5000  1309.124  2690.246  1096.2   1306.03   8238.923  72829.8                335.769                556.137
random_forest    192.704   5000 93322.204 32410.712 90377.7 189082.20 200407.246 503991.2              89011.210             198956.644


XGBoost apresentou P50 HTTP de **1.096 ms**; Random Forest, **90.378 ms**. Nesta bancada, a mediana do RF foi 82.4 vezes maior. A diferença não deve ser extrapolada para ARM ou para uma rede UAV.

## Lotes, throughput e recursos

In [3]:
batches = pd.read_csv(report_dir / "batch_http_latency.csv")
print(batches.round(3).to_string(index=False))

        model  batch_size  count  p50_ms  p95_ms  p99_ms  amortized_p50_us_per_row
      xgboost           1    150   1.142   1.400  22.485                  1142.050
      xgboost          32    150   1.803   2.089  24.132                    56.345
      xgboost         256    150   6.346  34.611  37.020                    24.789
      xgboost        1024    150  22.575  93.621  94.530                    22.046
random_forest           1    150  90.003 189.174 200.014                 90002.900
random_forest          32    150  97.907 174.675 200.259                  3059.606
random_forest         256    150 177.796 298.070 307.450                   694.514
random_forest        1024    150 194.702 304.976 383.792                   190.138


In [4]:
throughput = pd.read_csv(report_dir / "throughput.csv")
resources = pd.read_csv(report_dir / "resource_usage.csv")
print(throughput.round(3).to_string(index=False))
print()
print(resources.round(3).to_string(index=False))

        model  concurrency  mean_requests_per_second  std_requests_per_second
      xgboost            1                   767.533                   50.282
      xgboost            4                   446.796                   89.015
      xgboost            8                   487.952                   21.243
random_forest            1                    10.920                    0.104
random_forest            4                    10.831                    0.112
random_forest            8                    10.711                    0.099

        model  samples  cpu_percent_p50  cpu_percent_p95  cpu_percent_max  memory_mib_p50  memory_mib_p95  memory_mib_max sampling_errors
      xgboost       13            49.94            51.62            52.85           82.82          87.112           92.59              []
random_forest      493            50.07            51.29            53.81          464.30         474.580          478.60              []


O RF usou 5.2 vezes mais memória máxima observada e seu artefato foi 69.9 vezes maior. Concorrência adicional não aumentou o throughput do RF sob a quota de meio núcleo; no XGBoost, a melhor média também ocorreu com concorrência 1. Isso descreve esta implementação com acesso serializado ao modelo, não uma propriedade universal dos algoritmos.

## Fronteira qualidade–custo

In [5]:
tradeoff = pd.read_csv(report_dir / "quality_cost_tradeoff.csv")
print(tradeoff.round(6).to_string(index=False))

        model  s2_f1_macro_mean  s2_f1_macro_std  docker_http_p50_ms  docker_http_p95_ms  docker_http_p99_ms  best_mean_requests_per_second  best_throughput_concurrency  model_mib  memory_mib_max
      xgboost          0.951650         0.004264              1.0962             1.30603            8.238923                     767.532548                            1   2.756409           92.59
random_forest          0.950669         0.005136             90.3777           189.08220          200.407246                      10.919775                            1 192.704255          478.60


Na rodada de estabilidade S2, a diferença média de F1-macro XGBoost − RF foi **+0.000981**, pequena diante da variação entre folds. No benchmark Docker, porém, XGBoost combinou latência, tamanho e memória muito menores. A conclusão defensável é uma vantagem operacional do XGBoost nesta bancada, e não superioridade preditiva universal.

As métricas de qualidade vêm das predições OOF de S2. Os artefatos Docker foram reajustados em todos os dados somente para medição de sistemas e não recebem um novo escore preditivo nos dados de ajuste.

## Limites

O benchmark atende vetores de fluxo já calculados em loopback Docker Desktop/WSL2. Ele não inclui extração causal dos atributos, rádio, mobilidade, perda de rede, energia, bateria, hardware ARM ou tempo até detecção. Os limites de 0,5 CPU e 512 MiB são controles de cgroup, não uma simulação fiel de Raspberry Pi, Jetson ou drone físico.